In [55]:
using System;
using System.Threading;
string results = "";

public class DefiniteIntegral
{
    public double a {get;set;}
    public double b {get;set;}
    public Func<double, double> function {get;set;}
    public double step {get;set;}
    public int threadsnumber {get;set;}
    public static double Solve(double a, double b, Func<double, double> function, double step, int threadsnumber)
    {
        double totalsum = 0.0;
        var barrier = new Barrier(threadsnumber + 1);
        double sectionLength = (b-a)/threadsnumber;
        for (int i = 0; i < threadsnumber; i ++)
        {
            double localA = a + i * sectionLength;
            double localB = localA + sectionLength;
            if (localA > b)
            {
                localA = b;
            }
            if(localB > b)
            {
                localB = b;
            }
            new Thread(() =>
            {

                int n = (int)((localB-localA)/step);
                if (n < 1)
                {
                    n = 1;
                }
                double h = (localB - localA) / n;
                double sum = (function(localA) + function(localB)) / 2.0;
                for (int j  = 1; j < n; j++)
                {
                    double x = localA + j*h;
                    sum += function(x);
                }
                double localsum = sum * h;
                double oldValue;
                double newValue;
                while (true)
                {
                    oldValue = totalsum;
                    newValue = oldValue + localsum;
                    if (Interlocked.CompareExchange(ref totalsum, newValue, oldValue) == oldValue)
                    {
                        break;
                    }
                }
                barrier.SignalAndWait();
            }).Start();
            

        }
        barrier.SignalAndWait();
        return totalsum;
    }
    public static double SolveWithOneThread(double a, double b, Func<double,double> function, double step)
    {
        int n = (int) ((b-a)/step);
        if (n < 1)
        {
            n = 1;
        }
        double h = (b-a )/n;
        double sum = (function(a) + function(b)) / 2.0;
        for (int i = 1; i < n; i++)
        {
            double x = a + i*h;
            sum += function(x);
        }
        return sum * h;
    }
}

In [56]:
using System.Diagnostics;

double a = -100;
double b = 100;
Func<double, double> f = Math.Sin;


double[] steps = { 0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001 };
double target = 1e-4;
int numberofmeasure = 10;
bool optimalFounded = false;
results+= "Определение минимального размера шага, обеспечивающий оптимальную производительность с точностью 1e-4\n";
Console.WriteLine();
double optimalStep = steps[0];
double optimalTime = double.MaxValue;
double integral = 0;
double integralFourThreads = 0;
foreach (var step in steps)
{
    long totalTime = 0;
    long totalTimeFourThreads = 0;
    for (int j = 0; j < numberofmeasure; j++)
    {
        Stopwatch sw = Stopwatch.StartNew();
        integral = DefiniteIntegral.SolveWithOneThread(a,b,f,step);
        sw.Stop();
        totalTime+= sw.ElapsedMilliseconds;
    }
    for (int j = 0; j < numberofmeasure; j++)
    {
        Stopwatch swFourThreads = Stopwatch.StartNew();
        integralFourThreads = DefiniteIntegral.Solve(a,b,f,step,4);
        swFourThreads.Stop();
        totalTimeFourThreads+= swFourThreads.ElapsedMilliseconds;
    }
    double avgTime = totalTime/(double)numberofmeasure;
    double avgTimeFourThreads = totalTimeFourThreads/(double)numberofmeasure;
    double error = Math.Abs(integral);
    bool IsAccurate = false;
    if (error <= target) IsAccurate = true;
    if (IsAccurate && avgTime < optimalTime && avgTimeFourThreads < avgTime)
    {
        optimalTime = avgTime;
        optimalStep = step;
        optimalFounded = true;
    }
    results+=$"шаг: {step}, погрешность: {error}, ср. время: {avgTime} ms, погрешность меньше чем 1e-4: {IsAccurate}. ср. время на 4 потоках: {avgTimeFourThreads} ms \n";
    Console.WriteLine($"");
    if (avgTimeFourThreads >= avgTime)
    {
        results+="Среднее время на 1 потоке не может быть меньше или равно чем на 4\n";
    }
}
if (optimalFounded)
{
    results+=$"Оптимальный шаг найден - {optimalStep} \n";
}
else
{
    results+="Оптимальный шаг не найден\n";
}




In [ ]:
using System.Diagnostics;
results+="\n";
results+="Поиск оптимального кол-ва потоков\n";
List<double> avgTimes = new List<double>();
List<int> threads = new List<int>();
int maxthread = 16;
int numberofmeasure = 100;
double minTime = double.MaxValue;
int bestTread = 0;
double integral = 0;
for (int i = 1; i <= maxthread; i ++)
{
    double totaltime = 0;
    for (int j = 0; j < numberofmeasure; j ++)
    {
        Stopwatch sw = Stopwatch.StartNew();
        integral = DefiniteIntegral.Solve(a,b,f,optimalStep,i);
        sw.Stop();
        totaltime+=sw.ElapsedMilliseconds;
    }
    double avgTime = totaltime/(double)numberofmeasure;
    results+=$"Потоки: {i}, ср. время: {avgTime} ms\n";
    avgTimes.Add(avgTime);
    threads.Add(i);
    if (avgTime < minTime)
    {
        bestTread = i;
        minTime = avgTime;
    }
}
results+=$"Оптимальное кол-во потоков - {bestTread}\n";

In [58]:
#r "nuget: ScottPlot, 5.0.44"
using ScottPlot; 
var plt = new Plot();
var scatter = plt.Add.Scatter(
    avgTimes.Select(x => (double)x).ToArray(),
    threads.ToArray()
);
scatter.MarkerSize = 10;
scatter.LineWidth = 2;
scatter.Color = Colors.Green;
scatter.MarkerShape = MarkerShape.FilledCircle;
plt.Title("Зависимость времени выполнения от числа потоков");
plt.YLabel("Количество потоков");
plt.XLabel("Время выполнения (мс)");
plt.Grid.IsVisible = true;
string imagePath = "threads_performance.png";
plt.SavePng(imagePath, 800, 600);


Installed Packages ScottPlot, 5.0.44

In [59]:
using System.Text;
using System.IO;
double avgTime = 0;
double integral = 0;
results += $"\nСравнение результата на {bestTread} потоках с однопоточной реализацией\n";
results+= "\n";
results+= $"Лучший результат на {bestTread} потоках: {minTime}\n";
int numberofmeasure = 10;
double totaltime = 0;
for (int j = 0; j < numberofmeasure; j ++)
    {
        Stopwatch sw = Stopwatch.StartNew();
        integral = DefiniteIntegral.Solve(a,b,f,optimalStep,1);
        sw.Stop();
        totaltime+=sw.ElapsedMilliseconds;
    }
avgTime = totaltime/(double)numberofmeasure;
results+= $"Средний результат однопоточного варианта на 10 измерениях равен {avgTime}\n";
results+= $"Результат на {bestTread} потоках лучше на {((avgTime - minTime)/minTime)*100}%\n";
string filePath = Path.Combine(Directory.GetCurrentDirectory(), "report.txt");
File.WriteAllText(filePath, results, Encoding.UTF8);